# Luna AI — Fast QLoRA Fine-Tuning in Google Colab (Unsloth)

This notebook fine-tunes **Meta-Llama-3.1-8B-Instruct** (or Llama 3.3 70B) for **Luna AI** inside **LifeOS** using **Unsloth**.

### Luna's Hardened Invariants:
1. **Direct Answer First**: Sentence 1 delivers the recommendation or calculation (zero filler).
2. **Zero Emojis**: 100% clean typography.
3. **Mobile 3-Column Tables**: Maximum 3 columns (`| Item | Spec | Notes |`).
4. **Structured Blocks**: Emits ````workflow````, ````chart````, and silent ````json_action```` payloads.

In [ ]:
# 1. Install Unsloth and Dependencies (runs in ~1 minute)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets

In [ ]:
# 2. Load Base Model with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 3072
dtype = None # Auto detection (Float16 or Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 3. Add LoRA Adapters across all 7 linear layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 4. Upload or Download the 2,500 ChatML Dataset
# If running locally or on Colab, upload luna_chatml_dataset_2500.jsonl:
from datasets import load_dataset

dataset = load_dataset("json", data_files="luna_chatml_dataset_2500.jsonl", split="train")

def formatting_prompts_func(batch):
    convos = batch["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Initialize SFTTrainer & Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        max_steps=300,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="luna_checkpoints",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Test Model Response with Strict LifeOS Invariants
FastLanguageModel.for_inference(model)

test_prompt = "Give me a 35-minute push workout targeting upper chest and lateral delts with dumbbells."
messages = [
    {"role": "system", "content": "You are Luna, the hyper-intelligent executive AI for LifeOS. Follow all strict rules."},
    {"role": "user", "content": test_prompt}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 7. Save LoRA Adapter and GGUF for Deployment
model.save_pretrained_merged("luna_lora_adapter", tokenizer, save_method="lora")
print("LoRA Adapter saved to luna_lora_adapter folder. Ready to upload to Together AI, Fireworks AI, or run locally!")